# 01 · Graph Fundamentals

## What this notebook covers
Graphs are the natural representation for any data with pairwise relationships: social networks, molecules, knowledge bases, code dependency graphs, citation networks. This notebook covers the mathematical foundations and graph analysis with NetworkX.

## Graph vocabulary
| Term | Definition |
|------|-----------|
| **V** | Vertex (node) set |
| **E** | Edge set — pairs (u, v) |
| **Adjacency matrix A** | A[i,j]=1 if edge (i,j) exists |
| **Degree** | Number of edges per node |
| **Path** | Sequence of nodes connected by edges |
| **Component** | Maximal connected subgraph |
| **Diameter** | Longest shortest path in the graph |
| **Clustering coefficient** | Fraction of a node's neighbours that are also connected |

## Why graph structure matters for ML
Most ML assumes i.i.d. samples. Graph data is *relational* — the label of a node depends on its neighbours, and their neighbours, and so on. This is the core motivation for Graph Neural Networks (GNNs): aggregating neighbourhood information through message passing.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Three canonical graph types ────────────────────────────────────────────────
G_er  = nx.erdos_renyi_graph(50, 0.1, seed=42)       # random
G_ba  = nx.barabasi_albert_graph(50, 2, seed=42)      # scale-free (like the web)
G_ws  = nx.watts_strogatz_graph(50, 4, 0.3, seed=42)  # small-world (like social nets)

for name, G in [("Erdős–Rényi", G_er), ("Barabási–Albert", G_ba), ("Watts–Strogatz", G_ws)]:
    print(f"{name:20s}: nodes={G.number_of_nodes()}  edges={G.number_of_edges()}  "
          f"avg_degree={np.mean([d for _,d in G.degree()]):.2f}  "
          f"avg_clustering={nx.average_clustering(G):.3f}  "
          f"avg_shortest_path={nx.average_shortest_path_length(G):.2f}")

In [ ]:
# ── Centrality measures ────────────────────────────────────────────────────────
G = G_ba  # scale-free graph: has hubs

degree_c    = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)
pagerank    = nx.pagerank(G, alpha=0.85)
eigenvector = nx.eigenvector_centrality(G, max_iter=1000)

top5_degree = sorted(degree_c.items(), key=lambda x: -x[1])[:5]
top5_pr     = sorted(pagerank.items(), key=lambda x: -x[1])[:5]
print("Top-5 by degree centrality:", [(n, f"{v:.3f}") for n,v in top5_degree])
print("Top-5 by PageRank:          ", [(n, f"{v:.4f}") for n,v in top5_pr])

In [ ]:
# ── Adjacency matrix and Laplacian ────────────────────────────────────────────
A = nx.to_numpy_array(G_er)
D = np.diag(A.sum(axis=1))           # degree matrix
L = D - A                             # graph Laplacian
# Normalised Laplacian
D_inv_sqrt = np.diag(1.0 / np.sqrt(A.sum(axis=1) + 1e-8))
L_norm = np.eye(len(G_er)) - D_inv_sqrt @ A @ D_inv_sqrt

eigenvalues = np.linalg.eigvalsh(L_norm)
print(f"Normalised Laplacian eigenvalues (first 6): {eigenvalues[:6].round(4)}")
print(f"Fiedler value (algebraic connectivity): {eigenvalues[1]:.4f}  (>0 means connected)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, G) in zip(axes, [("Erdős–Rényi (random)", G_er), ("Barabási–Albert (scale-free)", G_ba), ("Watts–Strogatz (small-world)", G_ws)]):
    pos = nx.spring_layout(G, seed=42)
    deg = dict(G.degree())
    nx.draw(G, pos=pos, ax=ax, node_size=[deg[n]*30+30 for n in G.nodes()],
            node_color=[deg[n] for n in G.nodes()], cmap="Blues",
            edge_color="grey", alpha=0.8, with_labels=False)
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.savefig(f"{base}/01_graph_fundamentals.png", dpi=120)
plt.show()